# GridSearchCV (Random Forest)
**tabular › tuning_regression** · notebook `grid_search_rf`

**Kapan dipakai:** Ruang pencarian kecil dan ingin semua kombinasi dicoba (grid) agar hasilnya mudah dijelaskan di laporan.

**Jebakan umum:** Jumlah kombinasi berlipat ganda cepat: 2x3x2x2=24 kombinasi x 5 fold = 120 fit.

## 0. Setup
Import dan lokasi proyek. Tidak perlu diubah.

In [ ]:
import sys, pathlib, warnings
ROOT = next(p for p in [pathlib.Path.cwd().resolve(), *pathlib.Path.cwd().resolve().parents]
            if (p / "src" / "superdb").is_dir())
sys.path.insert(0, str(ROOT / "src"))
warnings.filterwarnings("ignore")
import numpy as np, pandas as pd, matplotlib.pyplot as plt
pd.set_option("display.max_columns", 50)
from superdb.common import *
from superdb.paths import out_dir

## 1. Konfigurasi
Ganti bagian ini saja saat memakai data lomba.

In [ ]:
TRAIN_PATH = ROOT / "data_latihan/tabular/regression/diamonds_train.csv"
TEST_PATH  = ROOT / "data_latihan/tabular/regression/diamonds_test.csv"      # None kalau tidak ada file test
TARGET     = "price"
ID_COL     = 'id'
DROP_COLS  = []
GROUP_COL  = None                      # isi bila ada grup agar tidak bocor
SCORING    = "neg_root_mean_squared_error"
SEED, N_SPLITS, VALID_SIZE = 42, 5, 0.2
MODEL_KEY  = "grid_search_rf"
TASK       = "regression"
KIND       = "tree"
OUT = out_dir("tabular", "tuning_regression", MODEL_KEY)
set_seed(SEED)

## 2. Baca data

In [ ]:
df = pd.read_csv(TRAIN_PATH)
test = pd.read_csv(TEST_PATH) if TEST_PATH else None
print(df.shape, None if test is None else test.shape)
display(df.head(3))

## 3. Fitur & split

In [ ]:
from superdb.tabular import detect_columns, prepare_features, build_pipeline, split_train_valid, cv_score
cols = detect_columns(df, target=TARGET, id_col=ID_COL, drop_cols=DROP_COLS)
X, NUM, CAT = prepare_features(df, cols)
y = df[TARGET].to_numpy(dtype=float)
groups = df[GROUP_COL] if GROUP_COL else None
Xtr, Xva, ytr, yva = split_train_valid(X, y, TASK, VALID_SIZE, SEED, groups)
GTR = None if groups is None else groups.iloc[Xtr.index]   # grup untuk CV
print(Xtr.shape, Xva.shape)

## 4. Model dasar & ruang pencarian
Nama parameter diawali `model__` karena model berada di dalam Pipeline.

In [ ]:
from sklearn.ensemble import RandomForestRegressor

base = RandomForestRegressor(n_jobs=-1, random_state=42)
param_dist = {"model__n_estimators": [200, 400], "model__max_depth": [6, 12, None],
              "model__min_samples_leaf": [1, 5], "model__max_features": ["sqrt", 0.6]}
SEARCH = "grid"

## 5. Jalankan pencarian

In [ ]:
from sklearn.model_selection import RandomizedSearchCV, GridSearchCV, StratifiedKFold, KFold
pipe0 = build_pipeline(base, NUM, CAT, kind=KIND)
cv = StratifiedKFold(N_SPLITS, shuffle=True, random_state=SEED) if TASK == "classification" else KFold(N_SPLITS, shuffle=True, random_state=SEED)
if SEARCH == "random":
    search = RandomizedSearchCV(pipe0, param_dist, n_iter=20, scoring=SCORING, cv=cv, random_state=SEED, n_jobs=-1)
else:
    search = GridSearchCV(pipe0, param_dist, scoring=SCORING, cv=cv, n_jobs=-1)
search.fit(Xtr, ytr)
print("skor CV terbaik:", round(search.best_score_, 4)); print(search.best_params_)
res = pd.DataFrame(search.cv_results_).sort_values("rank_test_score")
display(res[["rank_test_score", "mean_test_score", "std_test_score", "params"]].head(8))

## 6. Evaluasi pada validation

In [ ]:
pipe = search.best_estimator_

In [ ]:
pred = pipe.predict(Xva)
metrics = regression_metrics(yva, pred)
display(show_table(metrics))
plot_regression(yva, pred, path=OUT / "regression.png"); plt.show()

In [ ]:
save_json({"best_cv": search.best_score_, "best_params": search.best_params_, "valid": metrics}, OUT / "metrics.json")

## Latih ulang pada semua data → submission → simpan

In [ ]:
final = __import__('sklearn.base', fromlist=['clone']).clone(search.best_estimator_).fit(X, y)
if test is not None:
    Xte, _, _ = prepare_features(test, cols)
    Xte = Xte.reindex(columns=X.columns)
    ids = test[ID_COL] if ID_COL else np.arange(len(test))
    sub = make_submission(ids, final.predict(Xte), ID_COL or "id", TARGET, OUT / "submission.csv")
    display(sub.head(3))
save_model({"pipe": final, "cols": cols}, OUT / "model.joblib")
reload_pipe = load_model(OUT / "model.joblib")["pipe"]
assert np.allclose(reload_pipe.predict(X.head(50)), final.predict(X.head(50))), "model muat-ulang beda"
print("OK. hasil di", OUT)